[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashkanhszd/m8_data_science_module/blob/main/soluciones/M8_8.25_27_soluciones.ipynb)

# SOLUCIONES · M8_8.25–M8_8.27 · Bases de datos, estadística descriptiva y calidad de datos

**Máster en Hidrogeología y Modelación (UB) · Módulo 8: Herramientas y Data Science**

### Objetivos

- Organizar datos en tablas relacionadas y consultarlas con SQL.
- Describir variables con los estadísticos adecuados.
- Revisar la calidad de los datos de forma sistemática.
- Analizar relaciones entre variables con cautela.

## 0. Preparación

In [ ]:
# ==================================================================
# PREPARACIÓN DEL ENTORNO · ejecuta siempre esta celda en primer lugar
# ==================================================================
# · En Google Colab: descarga (clona) el repositorio del curso y se sitúa en él.
# · En un ordenador propio: busca la carpeta del repositorio.
# No hace falta entender cada línea todavía; la estructura se explica en M8_8.20.

import os
import subprocess
import sys
from pathlib import Path

URL_REPOSITORIO = "https://github.com/Ashkanhszd/m8_data_science_module.git"
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    ROOT = Path("/content/m8_data_science_module")
    if not ROOT.exists():
        print("Descargando el repositorio del curso...")
        subprocess.run(["git", "clone", "--depth", "1", URL_REPOSITORIO, str(ROOT)], check=True)
else:
    ROOT = None
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "data" / "input").exists():
            ROOT = carpeta
            break
    if ROOT is None:
        raise FileNotFoundError("No se encuentra data/input: abre el notebook desde la carpeta del repositorio.")

os.chdir(ROOT)
INPUT = ROOT / "data" / "input"          # datos originales (no se modifican)
DATABASE = ROOT / "data" / "database"    # base de datos SQLite
OUTPUT = ROOT / "data" / "output"        # resultados que generamos
for subcarpeta in ["tables", "figures", "logs"]:
    (OUTPUT / subcarpeta).mkdir(parents=True, exist_ok=True)

print("Entorno:", "Google Colab" if EN_COLAB else "ordenador local")
print("Carpeta del repositorio:", ROOT)
print("Python", sys.version.split()[0])

In [ ]:
import sqlite3

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("NumPy", np.__version__, "| pandas", pd.__version__, "| Matplotlib", matplotlib.__version__,
      "| SQLite", sqlite3.sqlite_version)

COLORES_ACUIFERO = {"Aluvial": "#2a78d6", "Carbonatado": "#eb6834", "Detritico": "#1baf7a"}

---
# M8_8.25 · Organización y gestión de datos científicos

## 1. ¿Por qué no basta con una hoja de cálculo?

Las hojas de cálculo son excelentes para mirar datos, pero como **almacén principal** de un proyecto dan problemas conocidos:

- se multiplican las copias (`datos_final_v3_bueno.xlsx`) y nadie sabe cuál es la buena;
- nada impide escribir `P1` en lugar de `P01`, mezclar unidades o dejar una celda con texto en una columna numérica;
- las celdas combinadas, los colores y los comentarios "guardan" información que un programa no puede leer;
- relacionar varias tablas (pozos, medidas, análisis) exige fórmulas frágiles;
- es difícil saber quién cambió qué y cuándo.

En M8_8.23 encontramos varios de estos problemas en nuestros datos. Una **base de datos relacional** no los elimina por arte de magia, pero permite **definir reglas** que los datos deben cumplir y **consultarlos** de forma reproducible.

## 2. La unidad de observación

Antes de analizar una tabla hay que responder a una pregunta: **¿qué representa exactamente una fila?**

| Tabla | Una fila es… | Identificador (clave) |
|---|---|---|
| `pozos` | un pozo | `id_pozo` |
| `estaciones` | una estación meteorológica | `id_estacion` |
| `precipitacion` | una estación en un mes | `id_estacion` + `fecha` |
| `mediciones` | un pozo en un mes | `id_pozo` + `fecha` |
| `hidroquimica` | un análisis de laboratorio | `id_pozo` + `fecha` |

Si sabemos qué debe ser una fila, podemos **comprobarlo**. Por ejemplo, cada pozo debería tener exactamente 60 mediciones (una por mes durante 5 años):

In [ ]:
mediciones = pd.read_csv(INPUT / "mediciones.csv", parse_dates=["fecha"])
pozos = pd.read_csv(INPUT / "pozos.csv")

print("¿id_pozo es único en pozos?", pozos["id_pozo"].is_unique)
mediciones.groupby("id_pozo").size()

La comprobación ya revela problemas: pozos con 59 o 61 filas y un pozo `P1` con una sola fila. **Definir bien la unidad de observación es la primera herramienta de control de calidad.**

## 3. Datos estructurados, semiestructurados y no estructurados

| Tipo | Características | Ejemplos en hidrogeología |
|---|---|---|
| **Estructurados** | Filas y columnas con tipos definidos | CSV de medidas, tablas SQL |
| **Semiestructurados** | Etiquetas y jerarquías flexibles | JSON de un sensor o de una API, XML, archivos de configuración |
| **No estructurados** | Sin estructura tabular previa | Informes en PDF, fotografías de testigos, notas de campo |

"No estructurado" no significa "sin información": significa que hay que **extraer** la estructura antes de analizarlo. El formato JSON es muy habitual. Así se vería la ficha de un pozo:

In [ ]:
import json

ficha = pozos.iloc[0].to_dict()
print(json.dumps(ficha, indent=2, ensure_ascii=False))

## 4. El modelo relacional de nuestros datos

En una base de datos **relacional**, la información se reparte en **tablas** que se relacionan mediante **claves**:

- una **clave primaria** (PK) identifica cada fila de una tabla de forma única;
- una **clave foránea** (FK) es una columna que contiene la clave primaria de **otra** tabla y crea la relación.

```text
 ┌──────────────────┐ 1        N ┌──────────────────┐ 1        N ┌──────────────────────┐
 │ estaciones       │────────────│ pozos            │────────────│ mediciones           │
 │ PK id_estacion   │            │ PK id_pozo       │            │ PK id_medicion       │
 │    nombre        │            │    acuifero      │            │ FK id_pozo           │
 │    x_utm, y_utm  │            │    x_utm, y_utm  │            │    fecha             │
 └──────────────────┘            │    cota_terreno_m│            │    profundidad_nivel_m│
          │ 1                    │ FK id_estacion   │            │    conductividad_uScm│
          │                      └──────────────────┘            └──────────────────────┘
          │ N                             │ 1
 ┌──────────────────┐                     │ N
 │ precipitacion    │            ┌──────────────────┐
 │ FK id_estacion   │            │ hidroquimica     │
 │    fecha         │            │ FK id_pozo       │
 │    precipitacion │            │    fecha, iones  │
 └──────────────────┘            └──────────────────┘
```

La **cardinalidad** describe cuántas filas de una tabla se relacionan con cuántas de otra:

| Relación | Ejemplo | Cómo se representa |
|---|---|---|
| **1 : N** (uno a muchos) | un pozo tiene muchas mediciones | la FK está en la tabla del lado "muchos" (`mediciones.id_pozo`) |
| **1 : 1** (uno a uno) | cada pozo tiene una ficha constructiva | la segunda tabla usa `id_pozo` como PK y FK a la vez |
| **N : M** (muchos a muchos) | un pozo pertenece a varias redes de control, y cada red incluye muchos pozos | una **tabla intermedia** `pozo_red (id_pozo, id_red)` |

> Observa que la precipitación está asociada a la **estación**, no al pozo. Si la guardáramos en `mediciones`, repetiríamos el mismo dato en todos los pozos de la misma estación, y un error habría que corregirlo muchas veces.

## 5. SQLite: una base de datos en un archivo

**SQLite** es una base de datos relacional completa guardada en **un solo archivo**, sin servidor. Python la incluye (`import sqlite3`). El repositorio contiene `data/database/hidrogeologia.sqlite` con las cinco tablas.

Para consultarla usamos **SQL** (*Structured Query Language*), el lenguaje estándar de las bases de datos relacionales. `pd.read_sql_query` ejecuta una consulta y devuelve el resultado como DataFrame. Para no repetir código, definimos una pequeña función:

In [ ]:
ruta_db = DATABASE / "hidrogeologia.sqlite"
conexion = sqlite3.connect(ruta_db)


def sql(consulta):
    """Ejecuta una consulta SQL sobre la base del curso y devuelve un DataFrame."""
    return pd.read_sql_query(consulta, conexion)


sql("SELECT name, type FROM sqlite_master WHERE type = 'table'")

El **esquema** de una tabla (columnas, tipos, restricciones) también se puede consultar:

In [ ]:
print(sql("SELECT sql FROM sqlite_master WHERE name = 'mediciones'").iloc[0, 0])

## 6. Consultas básicas: `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`

La estructura de una consulta se lee casi como una frase:

```sql
SELECT columnas            -- ¿qué columnas quiero?
FROM tabla                 -- ¿de qué tabla?
WHERE condición            -- ¿qué filas? (opcional)
ORDER BY columna DESC      -- ¿en qué orden? (opcional)
LIMIT 5;                   -- ¿cuántas filas como máximo? (opcional)
```

Por convención, las palabras de SQL se escriben en mayúsculas. `--` inicia un comentario.

In [ ]:
sql('''
SELECT id_pozo, fecha, conductividad_uScm
FROM mediciones
WHERE conductividad_uScm > 1300
ORDER BY conductividad_uScm DESC
LIMIT 5
''')

## 7. Unir tablas: `JOIN`

`JOIN` combina las filas de dos tablas que comparten un valor de clave. Es lo mismo que `merge` en pandas. Además, SQL puede **calcular** columnas nuevas, como la cota piezométrica.

In [ ]:
sql('''
SELECT
    m.id_pozo,
    m.fecha,
    p.acuifero,
    m.profundidad_nivel_m,
    p.cota_terreno_m - m.profundidad_nivel_m AS cota_piezometrica_m
FROM mediciones AS m
JOIN pozos AS p ON m.id_pozo = p.id_pozo
WHERE p.acuifero = 'Carbonatado'
ORDER BY m.id_pozo, m.fecha
LIMIT 6
''')

## 8. Resumir: `GROUP BY` y funciones de agregación

| Función | Resultado |
|---|---|
| `COUNT(col)` | número de valores no nulos |
| `AVG(col)` | media |
| `MIN(col)`, `MAX(col)` | extremos |
| `SUM(col)` | suma |

In [ ]:
sql('''
SELECT
    p.acuifero,
    COUNT(m.profundidad_nivel_m)        AS n_medidas,
    ROUND(AVG(m.profundidad_nivel_m), 2) AS profundidad_media_m,
    ROUND(AVG(m.conductividad_uScm), 0)  AS ce_media_uScm
FROM mediciones AS m
JOIN pozos AS p ON m.id_pozo = p.id_pozo
GROUP BY p.acuifero
''')

## 9. Integridad referencial: `LEFT JOIN` y restricciones

`JOIN` (o `INNER JOIN`) solo devuelve las filas que tienen pareja en las dos tablas. `LEFT JOIN` conserva **todas** las filas de la tabla de la izquierda y rellena con `NULL` (ausente) lo que no encuentra. Así se detectan mediciones "huérfanas":

In [ ]:
sql('''
SELECT m.id_medicion, m.id_pozo, m.fecha, p.acuifero
FROM mediciones AS m
LEFT JOIN pozos AS p ON m.id_pozo = p.id_pozo
WHERE p.id_pozo IS NULL
''')

SQLite puede comprobar **todas** las claves foráneas de la base de una vez:

In [ ]:
sql("PRAGMA foreign_key_check")

¿Por qué la base aceptó la fila de `P1`? Porque SQLite **no comprueba** las claves foráneas si no se le pide expresamente (`PRAGMA foreign_keys = ON`). Veamos qué pasa cuando sí se lo pedimos. Creamos una base de prueba **en memoria** con dos restricciones:

- `REFERENCES` (clave foránea): el pozo debe existir;
- `CHECK`: la profundidad no puede ser negativa.

In [ ]:
prueba = sqlite3.connect(":memory:")          # base temporal, solo en memoria
prueba.execute("PRAGMA foreign_keys = ON")

prueba.execute('''
CREATE TABLE pozos (
    id_pozo  TEXT PRIMARY KEY,
    acuifero TEXT NOT NULL
)''')
prueba.execute('''
CREATE TABLE mediciones (
    id_pozo             TEXT NOT NULL REFERENCES pozos(id_pozo),
    fecha               TEXT NOT NULL,
    profundidad_nivel_m REAL CHECK (profundidad_nivel_m >= 0),
    PRIMARY KEY (id_pozo, fecha)
)''')

prueba.execute("INSERT INTO pozos VALUES ('P01', 'Aluvial')")
prueba.execute("INSERT INTO mediciones VALUES ('P01', '2024-11-01', 7.8)")
print("Medición correcta insertada.")

intentos = [
    ("P1",  "2024-11-01", 7.8),     # el pozo no existe
    ("P01", "2024-12-01", -2.0),    # profundidad negativa
    ("P01", "2024-11-01", 7.9),     # la misma clave pozo-fecha repetida
]
for fila in intentos:
    try:
        prueba.execute("INSERT INTO mediciones VALUES (?, ?, ?)", fila)
        print("Insertada:", fila)
    except sqlite3.IntegrityError as error:
        print("RECHAZADA", fila, "→", error)
prueba.close()

> **Una base de datos bien diseñada impide que entren datos incoherentes**: identificadores inexistentes, valores imposibles y duplicados. Es mucho más barato evitar un error al introducir el dato que encontrarlo meses después.

## 10. Más allá de SQLite

- **PostgreSQL** es un gestor relacional robusto y multiusuario, muy usado en administraciones y empresas. **PostGIS** le añade tipos y funciones **espaciales** (puntos, polígonos, distancias).
- **GeoPackage** (`.gpkg`) es un formato geoespacial basado en SQLite. Es el formato por defecto de QGIS: una capa de QGIS puede ser una tabla de una base de datos.
- **NoSQL** agrupa bases no tabulares (documentos JSON, series temporales, grafos). Son útiles para datos de sensores de alta frecuencia o estructuras muy variables, pero no sustituyen al modelo relacional cuando importan las relaciones y la integridad.
- **DB Browser for SQLite** permite abrir `hidrogeologia.sqlite` y explorarla con una interfaz gráfica (ver la *Guía de instalación de SQLite* del curso). En clase lo veremos en directo.

## 11. Metadatos: los datos sobre los datos

Un número sin unidad, sin fecha, sin método y sin sistema de referencia **no es un dato científico**. El mínimo que debe acompañar a un conjunto de datos es: origen, fecha, unidades, sistema de referencia de las coordenadas, método de medida y límites de detección. En el curso, esa información está en el **diccionario de datos**:

In [ ]:
diccionario = pd.read_csv(INPUT / "diccionario_datos.csv")
diccionario.loc[diccionario["tabla"].isin(["pozos", "mediciones"])]

### Ejercicio 1 · Tu primera consulta SQL

Escribe una consulta que devuelva, para los pozos del acuífero **aluvial**, la **conductividad media de 2023** de cada pozo, ordenada de mayor a menor. Las columnas deben ser `id_pozo` y `ce_media`.

Las fechas están guardadas como texto (`'2023-07-01'`). Para extraer el año hay que usar una función de fechas de SQLite. Búscala en la documentación: https://www.sqlite.org/lang_datefunc.html (pista: `strftime`).

In [ ]:
consulta = '''
SELECT m.id_pozo, AVG(m.conductividad_uScm) AS ce_media
FROM mediciones AS m
JOIN pozos AS p ON m.id_pozo = p.id_pozo
WHERE p.acuifero = 'Aluvial' AND strftime('%Y', m.fecha) = '2023'
GROUP BY m.id_pozo
ORDER BY ce_media DESC
'''
ce_aluvial_2023 = sql(consulta)
ce_aluvial_2023

In [ ]:
# Comprobación
assert list(ce_aluvial_2023.columns) == ["id_pozo", "ce_media"], "Usa AS ce_media para nombrar la columna"
assert len(ce_aluvial_2023) == 5
assert ce_aluvial_2023.iloc[0]["id_pozo"] == "P03"
print("Correcto. P03 destaca en 2023: ¿por qué? Lo veremos en M8_8.27.")

---
# M8_8.26 · Estadística descriptiva

## 12. Población, muestra, variable

- **Población:** todo aquello sobre lo que queremos saber algo (por ejemplo, el estado del acuífero aluvial).
- **Muestra:** las observaciones de las que disponemos (60 medidas mensuales en 5 pozos).
- **Variable:** característica medida (profundidad del nivel, conductividad…).
- **Estadístico:** número que resume la muestra (media, mediana…). Lo usamos para **estimar** propiedades de la población.

Trabajaremos con los **datos tal como llegan**, con sus errores, para ver cómo afectan a cada estadístico. En M8_8.27 los corregiremos.

In [ ]:
datos = mediciones.merge(pozos, on="id_pozo", how="left")
datos.shape

## 13. Recuento, mínimo, máximo y rango

$$
R = x_{\max} - x_{\min}
$$

El rango es fácil de entender, pero depende solo de **dos** valores. Un único error lo cambia por completo.

In [ ]:
ce = datos["conductividad_uScm"]

print("Filas:", len(ce), "| Valores válidos:", ce.count(), "| Ausentes:", ce.isna().sum())
print("Mínimo:", ce.min(), "µS/cm")
print("Máximo:", ce.max(), "µS/cm")
print("Rango:", round(ce.max() - ce.min(), 1), "µS/cm")

Una conductividad mínima de ~1 µS/cm es inferior a la del agua destilada. **Los estadísticos más simples ya detectan errores.**

## 14. Media y mediana

La **media** aritmética es la suma de los valores dividida por su número:

$$
\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i
$$

La **mediana** es el valor central de los datos ordenados: la mitad de las observaciones está por debajo y la otra mitad por encima. Si $n$ es par, es la media de los dos valores centrales.

La media usa **todos** los valores y es sensible a los extremos. La mediana es **robusta**: apenas cambia aunque haya un valor absurdo.

Ejemplo con la profundidad del pozo **P11**, que contiene un valor de 270 m (el pozo tiene 80 m de profundidad):

In [ ]:
p11 = datos.loc[datos["id_pozo"] == "P11", "profundidad_nivel_m"]
p11_sin_error = p11[p11 < 80]

print(f"Media   con el error: {p11.mean():6.2f} m | sin el error: {p11_sin_error.mean():6.2f} m")
print(f"Mediana con el error: {p11.median():6.2f} m | sin el error: {p11_sin_error.median():6.2f} m")

Un solo valor erróneo entre 60 desplaza la media **4 m**, mientras que la mediana casi no cambia.

> **¿Cuál usar?** La media es adecuada para distribuciones simétricas y sin valores extremos, y cuando interesa el total (por ejemplo, la precipitación media anual). La mediana es preferible si la distribución es asimétrica o si puede haber valores extremos o erróneos. Calcular **ambas** y compararlas es una comprobación rápida: si difieren mucho, hay que mirar los datos.

## 15. Cuantiles y rango intercuartílico

Un **cuantil** $Q_p$ deja por debajo una proporción $p$ de los datos:

- $Q_1$ (percentil 25): deja por debajo el 25 %;
- $Q_2$ (percentil 50): la mediana;
- $Q_3$ (percentil 75): deja por debajo el 75 %.

El **rango intercuartílico** mide la dispersión de la mitad central de los datos:

$$
IQR = Q_3 - Q_1
$$

La regla habitual de los diagramas de caja marca como **valores alejados** los que quedan fuera de

$$
[\,Q_1 - 1.5\,IQR,\;\; Q_3 + 1.5\,IQR\,]
$$

> Un valor alejado **no es necesariamente un error**. La regla señala valores que conviene **revisar**, nada más.

In [ ]:
ce_p09 = datos.loc[datos["id_pozo"] == "P09", "conductividad_uScm"]

q1, q2, q3 = ce_p09.quantile([0.25, 0.50, 0.75])
iqr = q3 - q1
limite_inf, limite_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print(f"Q1 = {q1:.1f} | mediana = {q2:.1f} | Q3 = {q3:.1f} | IQR = {iqr:.1f} µS/cm")
print(f"Intervalo de revisión: {limite_inf:.1f} – {limite_sup:.1f} µS/cm")
print("Valores fuera del intervalo:")
datos.loc[(datos["id_pozo"] == "P09") & ~ce.between(limite_inf, limite_sup), ["id_pozo", "fecha", "conductividad_uScm"]]

## 16. Varianza y desviación típica

La **varianza** mide cuánto se alejan los valores de la media. Se calcula con las desviaciones al cuadrado, para que las positivas y las negativas no se anulen:

$$
s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2
$$

Se divide por $n-1$ (y no por $n$) porque estimamos la dispersión de la población a partir de una muestra. La varianza se expresa en unidades **al cuadrado** (m², (µS/cm)²), lo que dificulta su interpretación. Por eso se usa más la **desviación típica** (o estándar), que tiene las mismas unidades que la variable:

$$
s = \sqrt{s^2}
$$

In [ ]:
print(f"P11 · desviación típica con el error: {p11.std():.2f} m")
print(f"P11 · desviación típica sin el error: {p11_sin_error.std():.2f} m")
print(f"P11 · varianza sin el error: {p11_sin_error.var():.2f} m²")

La desviación típica pasa de 31 m a 1.5 m: es **aún más sensible** a los valores extremos que la media, porque eleva las desviaciones al cuadrado. La alternativa robusta es el IQR.

## 17. Histograma y diagrama de caja

- El **histograma** muestra la **forma** de la distribución: dónde se concentran los valores, si es simétrica o asimétrica y si hay varios grupos.
- El **diagrama de caja** (*boxplot*) resume mediana, cuartiles y valores alejados. Es ideal para **comparar grupos**.

Para estas figuras excluimos los dos valores claramente imposibles (se tratarán en M8_8.27):

In [ ]:
plausible = datos.loc[(datos["profundidad_nivel_m"] < datos["profundidad_pozo_m"]) & (datos["conductividad_uScm"] > 50)]
ce_plausible = plausible["conductividad_uScm"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.hist(ce_plausible, bins=40, color="#2a78d6", edgecolor="white")
ax1.axvline(ce_plausible.mean(), color="black", linewidth=2, label=f"media = {ce_plausible.mean():.0f}")
ax1.axvline(ce_plausible.median(), color="#eb6834", linewidth=2, linestyle="--", label=f"mediana = {ce_plausible.median():.0f}")
ax1.set_xlabel("Conductividad eléctrica (µS/cm)")
ax1.set_ylabel("Número de medidas")
ax1.set_title("Todos los pozos juntos")
ax1.legend()

orden = ["Aluvial", "Detritico", "Carbonatado"]
cajas = ax2.boxplot([plausible.loc[plausible["acuifero"] == a, "conductividad_uScm"] for a in orden],
                    patch_artist=True, medianprops={"color": "black"})
for caja, acuifero in zip(cajas["boxes"], orden):
    caja.set_facecolor(COLORES_ACUIFERO[acuifero])
ax2.set_xticks([1, 2, 3], orden)
ax2.set_ylabel("Conductividad eléctrica (µS/cm)")
ax2.set_title("Por acuífero")

fig.tight_layout()
plt.show()

El histograma global muestra **varios grupos**: mezcla aguas de tres acuíferos distintos. La media global (unos 870 µS/cm) **no describe ninguno de ellos**. El diagrama de caja lo aclara: el carbonatado está poco mineralizado, y el aluvial tiene más dispersión y varios valores muy altos.

> **Resumir primero por grupos con significado hidrogeológico** (acuífero, pozo, campaña) y solo después, si tiene sentido, globalmente.

## 18. Comparación entre grupos con pandas

In [ ]:
resumen = plausible.groupby("acuifero")["conductividad_uScm"].agg(
    n="count", media="mean", mediana="median", desv_tipica="std",
    q1=lambda x: x.quantile(0.25), q3=lambda x: x.quantile(0.75), minimo="min", maximo="max",
).round(1)
resumen

`describe()` calcula casi todo lo anterior de una vez. Ahora ya sabes qué significa cada fila:

In [ ]:
plausible.groupby("acuifero")["profundidad_nivel_m"].describe().round(2)

### Ejercicio 2 · ¿Qué estadísticos son robustos?

La conductividad de **P09** contiene un valor anotado en mS/cm en lugar de µS/cm (el valor ~1).

1. Calcula la media, la mediana, la desviación típica y el IQR de `ce_p09` **con** el valor erróneo. Guárdalos en un diccionario `con_error`.
2. Repite el cálculo **sin** ese valor (quédate con las conductividades mayores de 50 µS/cm) y guárdalos en `sin_error`.
3. Observa qué estadísticos cambian más de un 5 %.

In [ ]:
def resumen_robustez(serie):
    return {
        "media": serie.mean(),
        "mediana": serie.median(),
        "desv_tipica": serie.std(),
        "iqr": serie.quantile(0.75) - serie.quantile(0.25),
    }

con_error = resumen_robustez(ce_p09)
sin_error = resumen_robustez(ce_p09[ce_p09 > 50])

pd.DataFrame({"con error": con_error, "sin error": sin_error}).round(1)

In [ ]:
# Comprobación
cambio = {k: abs(con_error[k] - sin_error[k]) / sin_error[k] for k in con_error}
assert cambio["desv_tipica"] > 1, "La desviación típica debería cambiar muchísimo"
assert cambio["mediana"] < 0.05 and cambio["iqr"] < 0.05, "Mediana e IQR deberían cambiar muy poco"
print("Correcto. Mediana e IQR son robustos; media y, sobre todo, desviación típica no lo son.")

---
# M8_8.27 · Calidad de datos y relaciones entre variables

## 19. Control de calidad sistemático

La calidad de un dato se evalúa **respecto a un uso**. Un dato con un error de 10 cm puede servir para un inventario, pero no para calibrar un modelo. Estas son las comprobaciones mínimas, en el orden en que conviene hacerlas:

| # | Comprobación | Pregunta | Ejemplo en nuestros datos |
|---|---|---|---|
| 1 | **Estructura** | ¿Están todas las columnas? ¿Tienen el tipo correcto? | fechas leídas como texto |
| 2 | **Claves** | ¿Hay duplicados? ¿Todas las claves foráneas existen? | `P1`, filas repetidas |
| 3 | **Ausentes** | ¿Cuántos faltan? ¿Son aislados o hay huecos seguidos? | hueco de P07 |
| 4 | **Rangos físicos** | ¿Hay valores imposibles? | profundidad > pozo, lluvia negativa |
| 5 | **Unidades** | ¿Todos los valores están en la misma unidad? | CE en mS/cm |
| 6 | **Extremos** | ¿Hay valores inusuales? ¿Son errores o eventos reales? | CE alta en P03 |
| 7 | **Coherencia interna** | ¿Cuadran variables relacionadas? | balance iónico |

Recargamos los datos originales para trabajar desde cero:

In [ ]:
mediciones = pd.read_csv(INPUT / "mediciones.csv", parse_dates=["fecha"])
pozos = pd.read_csv(INPUT / "pozos.csv")
precipitacion = pd.read_csv(INPUT / "precipitacion.csv", parse_dates=["fecha"])

### 1 · Estructura

In [ ]:
columnas_requeridas = {"id_pozo", "fecha", "profundidad_nivel_m", "conductividad_uScm"}
print("Columnas que faltan:", columnas_requeridas - set(mediciones.columns))
print(mediciones.dtypes)

### 2 · Claves: duplicados e identificadores

- Un **duplicado exacto** repite la fila entera. Casi siempre se debe a haber copiado o cargado dos veces el mismo dato.
- Un **duplicado lógico** repite la **clave** (pozo + fecha) con valores **distintos**. Es más grave: uno de los dos es incorrecto, y no sabemos cuál.

In [ ]:
exactos = mediciones.duplicated(keep=False)
logicos = mediciones.duplicated(subset=["id_pozo", "fecha"], keep=False) & ~exactos

print("Filas en duplicados exactos:", exactos.sum())
print("Filas en duplicados lógicos:", logicos.sum())
mediciones.loc[logicos]

In [ ]:
# ¿Qué valor es más coherente? Miramos los meses vecinos de P05
mediciones.loc[(mediciones["id_pozo"] == "P05") & mediciones["fecha"].between("2024-03-01", "2024-07-01")]

In [ ]:
huerfanos = ~mediciones["id_pozo"].isin(pozos["id_pozo"])
mediciones.loc[huerfanos]

### 3 · Ausentes

No es lo mismo un ausente aislado que un **hueco** de varios meses seguidos. Un hueco puede indicar una avería del sensor y afecta más a los análisis temporales.

In [ ]:
ausentes = mediciones.loc[mediciones["profundidad_nivel_m"].isna(), ["id_pozo", "fecha"]]
ausentes

### 4 · Rangos físicos

Usamos el conocimiento del sistema: la profundidad del nivel no puede ser negativa (salvo en pozos surgentes) ni mayor que la del propio pozo, y la precipitación no puede ser negativa.

In [ ]:
con_pozo = mediciones.merge(pozos[["id_pozo", "profundidad_pozo_m"]], on="id_pozo", how="left")
imposible_prof = (con_pozo["profundidad_nivel_m"] < 0) | (con_pozo["profundidad_nivel_m"] > con_pozo["profundidad_pozo_m"])
print("Profundidades imposibles:")
print(con_pozo.loc[imposible_prof, ["id_pozo", "fecha", "profundidad_nivel_m", "profundidad_pozo_m"]])

print()
print("Precipitaciones negativas:")
print(precipitacion.loc[precipitacion["precipitacion_mm"] < 0])

### 5 · Unidades

Las aguas subterráneas dulces suelen tener conductividades de cientos a pocos miles de µS/cm. Un valor ~1000 veces menor que el resto del mismo pozo indica que se anotó en **mS/cm**.

In [ ]:
mediciones.loc[mediciones["conductividad_uScm"] < 50]

### 6 · Valores extremos: ¿error o evento real?

Aplicamos la regla del IQR **por pozo**, porque cada pozo tiene su propio rango normal:

In [ ]:
def fuera_de_rango_iqr(serie, factor=1.5):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (serie < q1 - factor * iqr) | (serie > q3 + factor * iqr)

ce_valida = mediciones["conductividad_uScm"] > 50     # excluimos el error de unidades ya detectado
alejada_15 = mediciones.groupby("id_pozo")["conductividad_uScm"].transform(fuera_de_rango_iqr)
print("Valores fuera de 1.5·IQR:", (alejada_15 & ce_valida).sum(), "de", ce_valida.sum())

mediciones["ce_alejada"] = mediciones.groupby("id_pozo")["conductividad_uScm"].transform(fuera_de_rango_iqr, factor=3)
mediciones.loc[mediciones["ce_alejada"] & ce_valida, ["id_pozo", "fecha", "profundidad_nivel_m", "conductividad_uScm"]]

Con el criterio de 1.5·IQR se marcan 15 valores de 720. Es lo esperable: en muchos datos siempre hay algunos que quedan fuera por simple variabilidad natural. Con un criterio más exigente, **3·IQR** ("muy alejados"), solo quedan cinco valores de **P03**, consecutivos (junio–octubre de 2023), en pleno verano de sequía y coincidiendo con los niveles más profundos del pozo, que es de riego. Un error aislado no suele repetirse cinco meses seguidos ni aumentar y disminuir de forma gradual. Lo más probable es un **evento real**: salinización por el bombeo intensivo y la falta de recarga.

> **Error, extremo y evento real no son lo mismo.** La estadística señala valores inusuales; la **decisión** requiere contexto hidrogeológico (patrón temporal, coherencia con otras variables, metadatos). Eliminar un evento real haría desaparecer justo lo que más interesa estudiar.

## 20. Tratamiento no destructivo

Principios:

1. **Nunca se modifican los datos originales.** Se trabaja sobre una **copia**.
2. Cada decisión queda **registrada**: qué se cambió, por qué y quién lo decidió.
3. Siempre que se pueda, se **marca** (bandera) en lugar de borrar.
4. Si no se puede justificar una corrección, el valor se marca como ausente, no se "adivina".

In [ ]:
limpios = mediciones.drop(columns="ce_alejada").copy()
limpios["bandera"] = ""
decisiones = []


def registrar(problema, accion, n_filas):
    decisiones.append({"problema": problema, "accion": accion, "filas_afectadas": int(n_filas)})


# 1. Duplicados exactos: se conserva una copia
n = limpios.duplicated().sum()
limpios = limpios.drop_duplicates().copy()
registrar("Duplicado exacto", "Se elimina la copia repetida", n)

# 2. Identificador mal escrito: P1 → P01 (en esa fecha falta el dato de P01, lo que apoya la corrección)
m = limpios["id_pozo"] == "P1"
limpios.loc[m, ["id_pozo", "bandera"]] = ["P01", "id_corregido"]
registrar("Identificador P1 inexistente", "Corregido a P01", m.sum())

# 3. Duplicado lógico de P05: los dos valores son plausibles → no se puede decidir; se deja un único registro sin valor
limpios = limpios.drop_duplicates(subset=["id_pozo", "fecha"], keep="first").copy()
m = (limpios["id_pozo"] == "P05") & (limpios["fecha"] == "2024-05-01")
limpios.loc[m, ["profundidad_nivel_m", "bandera"]] = [np.nan, "duplicado_logico"]
registrar("Duplicado lógico P05 2024-05", "Valor anulado (NaN): consultar al equipo de campo", m.sum())

# 4. Profundidad mayor que la del pozo: imposible → NaN
limpios = limpios.merge(pozos[["id_pozo", "profundidad_pozo_m"]], on="id_pozo", how="left")
m = limpios["profundidad_nivel_m"] > limpios["profundidad_pozo_m"]
limpios.loc[m, ["profundidad_nivel_m", "bandera"]] = [np.nan, "imposible"]
registrar("Profundidad mayor que la del pozo", "Valor anulado (NaN)", m.sum())
limpios = limpios.drop(columns="profundidad_pozo_m")

# 5. Conductividad en mS/cm: se convierte a µS/cm
m = limpios["conductividad_uScm"] < 50
limpios.loc[m, "conductividad_uScm"] = limpios.loc[m, "conductividad_uScm"] * 1000
limpios.loc[m, "bandera"] = "unidades_corregidas"
registrar("CE anotada en mS/cm", "Multiplicada por 1000", m.sum())

# 6. Evento real de P03: se conserva y se marca
m = (limpios["id_pozo"] == "P03") & limpios["fecha"].between("2023-06-01", "2023-10-01")
limpios.loc[m, "bandera"] = "evento_salinizacion"
registrar("CE alta P03 verano 2023", "Se conserva (evento real) y se marca", m.sum())

limpios = limpios.sort_values(["id_pozo", "fecha"]).reset_index(drop=True)
registro_decisiones = pd.DataFrame(decisiones)
registro_decisiones

In [ ]:
# Precipitación: el valor negativo se anula
precipitacion_limpia = precipitacion.copy()
precipitacion_limpia.loc[precipitacion_limpia["precipitacion_mm"] < 0, "precipitacion_mm"] = np.nan

# Comprobaciones finales y guardado en data/output (¡nunca en data/input!)
assert not limpios.duplicated(subset=["id_pozo", "fecha"]).any()
assert limpios["id_pozo"].isin(pozos["id_pozo"]).all()
assert limpios.groupby("id_pozo").size().eq(60).all(), "Cada pozo debería tener 60 meses"

limpios.to_csv(OUTPUT / "tables" / "mediciones_limpias.csv", index=False)
precipitacion_limpia.to_csv(OUTPUT / "tables" / "precipitacion_limpia.csv", index=False)
registro_decisiones.to_csv(OUTPUT / "tables" / "registro_decisiones_calidad.csv", index=False)
print("Datos limpios y registro de decisiones guardados en", OUTPUT / "tables")
limpios["bandera"].value_counts()

## 21. Datos censurados: valores por debajo del límite de detección

En hidroquímica es habitual que el laboratorio informe `<0.5`: la concentración es **menor que el límite de detección** (LD), pero no sabemos cuánto. Es un **dato censurado**: sabemos algo (que es pequeño), pero no el valor exacto.

> Convertir `<0.5` en `NaN` **pierde información**: estaríamos diciendo que no sabemos nada. Convertirlo en `0` o en `0.5` introduce un sesgo. Lo más transparente es **separar el valor y una bandera**, y decidir después según el análisis.

In [ ]:
quimica = pd.read_csv(INPUT / "hidroquimica.csv", parse_dates=["fecha"])
print("Tipo de la columna NO3_mgL:", quimica["NO3_mgL"].dtype)   # texto, por culpa de los '<0.5'

texto_no3 = quimica["NO3_mgL"].astype(str)
quimica["NO3_censurado"] = texto_no3.str.startswith("<")
quimica["NO3_valor"] = pd.to_numeric(texto_no3.str.replace("<", "", regex=False))   # el número (el LD si está censurado)

quimica.loc[quimica["NO3_censurado"], ["id_pozo", "fecha", "NO3_mgL", "NO3_valor", "NO3_censurado"]]

Una práctica sencilla y habitual para cálculos exploratorios es sustituir los censurados por **LD/2**, dejando constancia de ello. Para análisis rigurosos existen métodos estadísticos específicos para datos censurados.

In [ ]:
quimica["NO3_mgL_num"] = np.where(quimica["NO3_censurado"], quimica["NO3_valor"] / 2, quimica["NO3_valor"])
quimica.groupby(quimica["id_pozo"].map(pozos.set_index("id_pozo")["acuifero"]))["NO3_mgL_num"].median()

## 22. Coherencia interna: el balance iónico

En un análisis químico completo, el agua es eléctricamente neutra: la suma de cationes debe igualar la suma de aniones, expresadas en **miliequivalentes por litro** (meq/L). Para convertir mg/L en meq/L se divide por el **peso equivalente** (masa molar / carga):

$$
C\,[\mathrm{meq/L}] = \frac{C\,[\mathrm{mg/L}]}{\text{peso equivalente}}
$$

El **error de balance iónico** (EBI) es:

$$
EBI\,(\%) = 100 \cdot \frac{\sum \text{cationes} - \sum \text{aniones}}{\sum \text{cationes} + \sum \text{aniones}}
$$

Como criterio habitual, un $|EBI|$ menor del 5 % es aceptable, y uno mayor del 10 % indica un problema en el análisis (un ion mal medido o un ion importante que falta).

| Ion | Ca²⁺ | Mg²⁺ | Na⁺ | K⁺ | HCO₃⁻ | Cl⁻ | SO₄²⁻ | NO₃⁻ |
|---|---|---|---|---|---|---|---|---|
| Peso equivalente (mg/meq) | 20.04 | 12.15 | 22.99 | 39.10 | 61.02 | 35.45 | 48.03 | 62.00 |

### Ejercicio 3 · Balance iónico

1. Calcula la suma de cationes (`suma_cationes`) y de aniones (`suma_aniones`) en meq/L para cada análisis de `quimica`. Usa `NO3_mgL_num` para el nitrato.
2. Calcula el EBI en % en una nueva columna `quimica["EBI"]`.
3. Muestra los análisis con `|EBI| > 10`.

In [ ]:
PESO_EQUIVALENTE = {"Ca": 20.04, "Mg": 12.15, "Na": 22.99, "K": 39.10,
                    "HCO3": 61.02, "Cl": 35.45, "SO4": 48.03, "NO3": 62.00}

In [ ]:
suma_cationes = (quimica["Ca_mgL"] / PESO_EQUIVALENTE["Ca"]
                 + quimica["Mg_mgL"] / PESO_EQUIVALENTE["Mg"]
                 + quimica["Na_mgL"] / PESO_EQUIVALENTE["Na"]
                 + quimica["K_mgL"] / PESO_EQUIVALENTE["K"])
suma_aniones = (quimica["HCO3_mgL"] / PESO_EQUIVALENTE["HCO3"]
                + quimica["Cl_mgL"] / PESO_EQUIVALENTE["Cl"]
                + quimica["SO4_mgL"] / PESO_EQUIVALENTE["SO4"]
                + quimica["NO3_mgL_num"] / PESO_EQUIVALENTE["NO3"])

quimica["EBI"] = 100 * (suma_cationes - suma_aniones) / (suma_cationes + suma_aniones)

quimica.loc[quimica["EBI"].abs() > 10, ["id_pozo", "fecha", "EBI"]]

In [ ]:
# Comprobación
malos = quimica.loc[quimica["EBI"].abs() > 10]
assert len(malos) == 1 and malos.iloc[0]["id_pozo"] == "P08"
assert (quimica["EBI"].abs() <= 10).sum() == 47
print(f"Correcto. El análisis de P08 ({malos.iloc[0]['fecha']:%Y-%m}) tiene un EBI de {malos.iloc[0]['EBI']:.0f} %:")
print("  exceso de cationes, probablemente un error en el sodio. Se debería pedir al laboratorio que lo revise.")

## 23. Relaciones entre dos variables

### Diagrama de dispersión y coeficientes de correlación

El primer paso es **siempre** mirar el diagrama de dispersión. Después, un coeficiente puede resumir la relación:

**Pearson** mide la asociación **lineal**:

$$
r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2}\,\sqrt{\sum (y_i - \bar{y})^2}}
$$

**Spearman** aplica la misma fórmula a los **rangos** (la posición de cada valor ordenado) y mide si la relación es **monótona** (siempre creciente o siempre decreciente), aunque no sea lineal. Sin empates:

$$
\rho = 1 - \frac{6\sum d_i^2}{n(n^2 - 1)}
$$

donde $d_i$ es la diferencia entre los rangos de $x_i$ y de $y_i$.

Ambos valen entre −1 (relación inversa perfecta) y +1 (directa perfecta). Un valor cercano a 0 indica que no hay relación **de ese tipo**, pero puede haber otras.

Comparemos los coeficientes calculados con los datos **originales** y con los **limpios**:

In [ ]:
originales = mediciones.merge(pozos, on="id_pozo", how="left")
depurados = limpios.merge(pozos, on="id_pozo", how="left")

variables = ["profundidad_nivel_m", "conductividad_uScm"]
comparacion = pd.DataFrame({
    "Pearson": [originales[variables].corr("pearson").iloc[0, 1], depurados[variables].corr("pearson").iloc[0, 1]],
    "Spearman": [originales[variables].corr("spearman").iloc[0, 1], depurados[variables].corr("spearman").iloc[0, 1]],
}, index=["datos originales", "datos limpios"]).round(2)
comparacion

Con los datos originales, Pearson (−0.52) y Spearman (−0.80) discrepan mucho: **dos valores erróneos bastan para distorsionar Pearson**, mientras que Spearman, al trabajar con rangos, apenas se inmuta.

Con los datos limpios parece haber una relación negativa fuerte: *cuanto más profundo el nivel, menor conductividad*. Pero **mira el gráfico antes de creértelo**:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for acuifero, grupo in depurados.groupby("acuifero"):
    ax1.scatter(grupo["profundidad_nivel_m"], grupo["conductividad_uScm"], s=10, alpha=0.6,
                color=COLORES_ACUIFERO[acuifero], label=acuifero)
r_global = depurados[variables].corr().iloc[0, 1]
ax1.set_title(f"Todos los pozos: r = {r_global:.2f}")
ax1.set_xlabel("Profundidad del nivel (m)")
ax1.set_ylabel("Conductividad eléctrica (µS/cm)")
ax1.legend(title="Acuífero")
ax1.grid(alpha=0.3)

correlaciones = {}
for acuifero, grupo in depurados.groupby("acuifero"):
    correlaciones[acuifero] = grupo[variables].corr().iloc[0, 1]
ax2.bar(list(correlaciones), list(correlaciones.values()),
        color=[COLORES_ACUIFERO[a] for a in correlaciones])
ax2.axhline(0, color="black", linewidth=0.8)
ax2.axhline(r_global, color="#52514e", linestyle="--", linewidth=1)
ax2.text(2.45, r_global, "r global", va="bottom", ha="right", color="#52514e")
ax2.set_ylim(-1, 1)
ax2.set_ylabel("Coeficiente de Pearson")
ax2.set_title("Dentro de cada acuífero")
for i, valor in enumerate(correlaciones.values()):
    ax2.text(i, valor + 0.04, f"{valor:.2f}", ha="center")

fig.tight_layout()
plt.show()

**Dentro de cada acuífero, la relación es positiva**: cuando el nivel baja, el agua se concentra y la conductividad aumenta, algo coherente con la sequía. La correlación global negativa se debe **solo** a que el acuífero carbonatado es a la vez profundo y poco mineralizado. El **acuífero** es una variable de confusión, y el fenómeno se conoce como **paradoja de Simpson**.

> **Correlación no implica causalidad.** Una asociación puede deberse a una tercera variable (acuífero, litología, época del año), a la mezcla de grupos, a unos pocos valores influyentes o al azar. Para hablar de causa hace falta un **mecanismo físico** defendible y evidencia adicional.

## 24. Dependencia temporal: el retardo de la recarga

En M8_8.24 vimos que el nivel no sube el mismo mes en que llueve. Podemos **medir** ese retardo correlacionando la precipitación de hace $k$ meses con la **variación** mensual del nivel. `shift(k)` desplaza una serie $k$ posiciones hacia abajo, es decir, trae el valor de $k$ meses antes.

In [ ]:
serie = (depurados
         .merge(precipitacion_limpia, on=["id_estacion", "fecha"], how="left")
         .sort_values(["id_pozo", "fecha"]))

# Subida del nivel respecto al mes anterior (positiva = el nivel sube = la profundidad disminuye)
serie["subida_nivel_m"] = -serie.groupby("id_pozo")["profundidad_nivel_m"].diff()

filas = []
for retardo in range(0, 5):
    serie["lluvia_retrasada"] = serie.groupby("id_pozo")["precipitacion_mm"].shift(retardo)
    for acuifero, grupo in serie.groupby("acuifero"):
        r = grupo[["lluvia_retrasada", "subida_nivel_m"]].corr().iloc[0, 1]
        filas.append({"retardo_meses": retardo, "acuifero": acuifero, "r": r})

tabla_retardos = pd.DataFrame(filas).pivot(index="retardo_meses", columns="acuifero", values="r").round(2)
tabla_retardos

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for acuifero in tabla_retardos.columns:
    ax.plot(tabla_retardos.index, tabla_retardos[acuifero], marker="o", linewidth=2,
            color=COLORES_ACUIFERO[acuifero], label=acuifero)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(tabla_retardos.index)
ax.set_xlabel("Retardo (meses)")
ax.set_ylabel("r (lluvia retrasada vs. subida del nivel)")
ax.set_title("¿Cuántos meses tarda la lluvia en llegar al acuífero?")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print("Retardo con máxima correlación:", tabla_retardos.idxmax().to_dict())

El **aluvial**, somero y conectado con la superficie, responde en **1 mes**. El **carbonatado** y el **detrítico** tardan unos **2 meses**. Es un resultado con sentido físico: el tiempo de tránsito por la zona no saturada.

> **Dependencia temporal y espacial.** Las 60 medidas de un pozo **no son independientes**: el nivel de este mes se parece mucho al del mes pasado. Lo mismo ocurre con pozos cercanos. Muchas técnicas estadísticas suponen observaciones independientes. Si no lo son, los resultados parecen más seguros de lo que realmente son. Tenlo muy presente en el *machine learning*.

## 25. Puente hacia el *machine learning* (M8_8.37–42)

Un modelo de *machine learning* aprende una relación a partir de datos. Para saber si funciona, los datos se dividen en tres conjuntos:

| Conjunto | Para qué sirve |
|---|---|
| **Entrenamiento** | ajustar el modelo |
| **Validación** | elegir entre modelos o configuraciones |
| **Prueba** (*test*) | evaluar el resultado final con datos que el modelo no ha visto nunca |

> Por la dependencia temporal, **no** conviene repartir al azar los meses de un mismo pozo entre entrenamiento y prueba: el modelo "vería" meses casi idénticos a los de prueba y parecería mejor de lo que es. Según la pregunta, se separa por **pozo**, por **zona** o por **periodo** (por ejemplo, entrenar con 2021–2024 y probar con 2025).

**Métricas de error** habituales, con $y_i$ el valor observado, $\hat{y}_i$ el predicho y $n$ el número de datos:

$$
MAE = \frac{1}{n}\sum |y_i - \hat{y}_i| \qquad
RMSE = \sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2} \qquad
R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}
$$

- **MAE**: error medio, en las unidades de la variable.
- **RMSE**: similar, pero penaliza más los errores grandes.
- **R²**: fracción de la variabilidad explicada (1 = perfecto; 0 = no mejora a predecir siempre la media).

La figura siguiente ilustra el mayor riesgo de un modelo: **ajustarse demasiado** a los datos de entrenamiento (*overfitting*) o **demasiado poco** (*underfitting*). Los tres "modelos" son polinomios de distinto grado ajustados con `np.polyfit` a los mismos 12 puntos de entrenamiento, y evaluados con 10 puntos de validación que no se usaron en el ajuste.

In [ ]:
rng = np.random.default_rng(23)
x = np.linspace(0, 10, 300)
patron = 7 + 0.45 * x + 1.8 * np.sin(0.9 * x)
x_ent = np.linspace(0.5, 9.5, 12)
y_ent = 7 + 0.45 * x_ent + 1.8 * np.sin(0.9 * x_ent) + rng.normal(0, 0.5, x_ent.size)
x_val = np.sort(rng.uniform(1.5, 8.5, 10))
y_val = 7 + 0.45 * x_val + 1.8 * np.sin(0.9 * x_val) + rng.normal(0, 0.5, x_val.size)

fig, ejes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for ax, grado, titulo in zip(ejes, [1, 4, 11], ["Underfitting (grado 1)", "Ajuste adecuado (grado 4)", "Overfitting (grado 11)"]):
    coef = np.polyfit(x_ent, y_ent, grado)
    ajuste = np.polyval(coef, x)
    rmse_ent = np.sqrt(np.mean((np.polyval(coef, x_ent) - y_ent) ** 2))
    rmse_val = np.sqrt(np.mean((np.polyval(coef, x_val) - y_val) ** 2))
    ax.plot(x, patron, color="#52514e", linestyle="--", linewidth=1.5, label="patrón real")
    ax.plot(x, ajuste, color="#eb6834", linewidth=2, label="modelo")
    ax.scatter(x_ent, y_ent, color="#2a78d6", s=30, zorder=3, label="entrenamiento")
    ax.scatter(x_val, y_val, color="#1baf7a", marker="s", s=30, zorder=3, label="validación")
    ax.set_ylim(4, 16)
    ax.set_title(f"{titulo}\nRMSE entren. = {rmse_ent:.2f} · RMSE valid. = {rmse_val:.2f}")
    ax.set_xlabel("variable explicativa")
ejes[0].set_ylabel("variable a predecir")
ejes[2].legend(loc="upper left")
fig.tight_layout()
plt.show()

El modelo con *overfitting* tiene el **menor error de entrenamiento** pero un **error de validación mayor**: ha memorizado el ruido. El objetivo es **generalizar**, es decir, predecir bien datos nuevos.

## Síntesis

- **Organización:** define la unidad de observación y las claves. Una base relacional con restricciones impide errores como `P1`, valores negativos o duplicados.
- **SQL:** `SELECT … FROM … JOIN … WHERE … GROUP BY … ORDER BY`. `LEFT JOIN` y `PRAGMA foreign_key_check` detectan huérfanos.
- **Estadística:** la media y la desviación típica son sensibles a los errores; la mediana y el IQR son robustos. Resume **por grupos** con sentido hidrogeológico.
- **Calidad:** estructura → claves → ausentes → rangos → unidades → extremos → coherencia. El tratamiento es **no destructivo y documentado**. No todo valor extremo es un error.
- **Censurados:** separa valor y bandera; no los conviertas en ausentes.
- **Relaciones:** mira el gráfico antes del coeficiente; desconfía de las correlaciones globales entre grupos distintos; ten en cuenta los retardos y la dependencia temporal.

En **M8_8.31–33** convertiremos este control de calidad en **funciones reutilizables** y en un **flujo automático**.

In [ ]:
conexion.close()